# 2 - Intégration des différents composants

Ce notebook illustre un **workflow complet** en combinant l'ensemble des composants du package :
construction de la base, mise à jour, suppression, audit d'intégrité, maintenance physique et voyage dans le temps.

Il constitue le point d'entrée recommandé pour comprendre comment les classes interagissent dans un scénario de production.

### Table des matières

0. [Importation des modules](#section_0)
1. [Création des données synthétiques](#section_1)
2. [Construction de la base de données](#section_2)
   - [Connexion et initialisation](#section_2_1)
   - [Construction du schéma](#section_2_2)
   - [Vérification de l'état initial](#section_2_3)
3. [Mise à jour de la base de données](#section_3)
   - [Mise à jour sans impact sur les tables de dimension](#section_3_1)
   - [Ajout de nouvelles observations](#section_3_2)
4. [Suppression d'observations](#section_4)
   - [Suppression simple sans impact dimensionnel](#section_4_1)
   - [Suppression avec nettoyage dimensionnel](#section_4_2)
5. [Audit de la base de données](#section_5)
   - [Audit basique](#section_5_1)
   - [Audit standard](#section_5_2)
   - [Audit complet](#section_5_3)
6. [Maintenance physique (DuckLake)](#section_6)
   - [Opérations individuelles](#section_6_1)
   - [Maintenance complète](#section_6_2)
7. [Voyage dans le temps (time-travel)](#section_7)
   - [Consultation d'un snapshot par version](#section_7_1)
   - [Consultation d'un snapshot par horodatage](#section_7_2)

## 0. Importation des modules <a id="section_0"></a>

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import os
import shutil
import sys
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Ajout du chemin racine du projet
sys.path.append('..')

# Connexion DuckLake
from dt_ducklake_manager.connection import DuckLakeConnector

# Construction du schéma
from dt_ducklake_manager.schema import DuckLakeTablesBuilder

# Opérations sur les données
from dt_ducklake_manager.operations import (
    DatabaseUpdater,
    DatabaseDeleter
)

# Maintenance et audit
from dt_ducklake_manager.maintenance import (
    DatabaseAuditor,
    ValidationLevel,
    DuckLakeMaintenance,
)

## 1. Création des données synthétiques <a id="section_1"></a>

In [ ]:
# Initialisation du générateur aléatoire pour la reproductibilité
np.random.seed(42)

# Paramètres du jeu de données synthétiques
N_ROWS = 200
CATEGORICAL_THRESHOLD = 8
START_DATE = datetime(2024, 1, 1)

# Modalités des colonnes catégorielles
indicators   = ['temperature', 'humidity', 'pressure', 'wind_speed']
countries    = ['France', 'Germany', 'Italy', 'Spain']
kinds        = ['forecast', 'observation']
models       = ['model_A', 'model_B', 'model_C']
trainings    = ['train_v1', 'train_v2']
horizons     = [1, 3, 7, 14]

# Construction du DataFrame initial
data_list = []
for i in range(N_ROWS):
    date = START_DATE + timedelta(days=i % 180)
    row = {
        'indicator': np.random.choice(indicators),
        'country':   np.random.choice(countries),
        'kind':      np.random.choice(kinds),
        'model':     np.random.choice(models),
        'training':  np.random.choice(trainings),
        'horizon':   np.random.choice(horizons),
        'date':      date,
        'value':          np.random.uniform(10, 100),
        'lower_bound':    np.random.uniform(5,  50) if np.random.random() > 0.4 else None,
        'upper_bound':    np.random.uniform(50, 150) if np.random.random() > 0.4 else None,
        'quality_score':  np.random.uniform(0, 1),
    }
    data_list.append(row)

df_origin = pd.DataFrame(data_list)
df_origin['date'] = pd.to_datetime(df_origin['date'])

# Clé primaire composite
PK_COLUMNS = ['indicator', 'country', 'kind', 'model', 'training', 'horizon', 'date']
df_origin = df_origin.drop_duplicates(subset=PK_COLUMNS, keep='first').reset_index(drop=True)

# Labels des colonnes (utilisés dans les métadonnées)
LABELS = {
    'indicator':     'Indicateur',
    'country':       'Pays',
    'kind':          'Type',
    'model':         'Modèle',
    'training':      'Entraînement',
    'horizon':       'Horizon',
    'date':          'Date',
    'value':         'Valeur',
    'lower_bound':   'Borne inférieure',
    'upper_bound':   'Borne supérieure',
    'quality_score': 'Score de qualité',
}

# Chemins du catalogue et des données DuckLake
CATALOG_PATH = os.path.join('../outputs', 'workflow_integration.ducklake')
DATA_PATH    = os.path.join('../outputs', 'workflow_integration_data/')

# Affichage du résumé du jeu de données
print(f"Nombre de lignes après déduplication : {len(df_origin)}")
print(f"Seuil catégoriel : {CATEGORICAL_THRESHOLD}")
print(f"Catalogue : {CATALOG_PATH}")
df_origin.head()

## 2. Construction de la base de données <a id="section_2"></a>

### 2.1. Connexion et initialisation <a id="section_2_1"></a>

In [ ]:
# Suppression d'un éventuel état résiduel pour garantir un départ propre
for suffix in ['', '.wal']:
    path_to_remove = CATALOG_PATH + suffix
    if os.path.exists(path_to_remove):
        os.remove(path_to_remove)
        print(f"Fichier supprimé : {path_to_remove}")
if os.path.exists(DATA_PATH):
    shutil.rmtree(DATA_PATH)
    print(f"Répertoire supprimé : {DATA_PATH}")

# Ouverture de la connexion DuckLake en lecture-écriture
conn = DuckLakeConnector(CATALOG_PATH, DATA_PATH).connect()
print("Connexion DuckLake ouverte.")

### 2.2. Construction du schéma <a id="section_2_2"></a>

In [ ]:
# Initialisation du builder avec clé primaire composite
# Le seuil catégoriel = 8 : les colonnes avec ≤ 8 valeurs uniques génèrent une table de dimension
builder = DuckLakeTablesBuilder(
    df=df_origin,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    primary_keys=PK_COLUMNS,
    connection=conn,
)

# Construction du schéma complet (fact_table + metadata + tables de dimension)
builder.build_schema(column_labels=LABELS)

# Vérification rapide du nombre de lignes insérées
n = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
print(f"Lignes dans fact_table après construction : {n}")

### 2.3. Vérification de l'état initial <a id="section_2_3"></a>

In [ ]:
# Affichage du schéma (tables créées, colonnes, types)
builder.display_schema()

In [ ]:
# Lecture des métadonnées : statut catégoriel, type Python, nombre de modalités
df_meta = conn.execute("SELECT * FROM metadata").fetchdf()
display(df_meta[['name', 'label', 'python_type', 'is_categorical']])

In [ ]:
# Inspection du contenu des tables de dimension générées
# Les colonnes catégorielles (≤ 8 modalités) ont chacune leur propre table de dimension
all_tables = [row[0] for row in conn.execute("SHOW TABLES").fetchall()]
dim_tables = sorted([t for t in all_tables if t.startswith('dim_')])
print(f"Tables de dimension créées : {dim_tables}")

for table in dim_tables:
    print(f"\n--- {table} ---")
    display(conn.execute(f"SELECT * FROM {table} ORDER BY value").fetchdf())

## 3. Mise à jour de la base de données <a id="section_3"></a>

Le `DatabaseUpdater` effectue un **upsert** : les lignes existantes (identifiées par la clé primaire) sont mises à jour, les nouvelles lignes sont insérées.

In [ ]:
# Initialisation du composant de mise à jour
# La connexion est partagée avec le builder — les opérations portent sur le même état
updater = DatabaseUpdater(
    connection=conn,
    categorical_threshold=CATEGORICAL_THRESHOLD,
)

### 3.1. Mise à jour sans impact sur les tables de dimension <a id="section_3_1"></a>

Modification des colonnes numériques (`value`, `quality_score`) de 10 lignes existantes.
Le statut catégoriel des colonnes n'est pas affecté : aucune table de dimension n'est créée ni supprimée.

In [ ]:
# Reconstruction de 10 lignes existantes avec leur clé primaire complète
# Les colonnes catégorielles stockées sous forme d'ID dans fact_table sont
# récupérées via jointure avec les tables de dimension
sample_rows = conn.execute("""
    SELECT
        i.label  AS indicator,
        c.label  AS country,
        k.label  AS kind,
        m.label  AS model,
        tr.label AS training,
        f.horizon,
        f.date,
        f.value,
        f.lower_bound,
        f.upper_bound,
        f.quality_score
    FROM fact_table f
    JOIN dim_indicator i  ON f.indicator = i.value
    JOIN dim_country   c  ON f.country   = c.value
    JOIN dim_kind      k  ON f.kind      = k.value
    JOIN dim_model     m  ON f.model     = m.value
    JOIN dim_training  tr ON f.training  = tr.value
    LIMIT 10
""").fetchdf()

# Modification des colonnes numériques uniquement
update_31 = sample_rows.copy()
update_31['value']         = update_31['value']         * 1.10   # Augmentation de 10 %
update_31['quality_score'] = update_31['quality_score'] * 0.95   # Légère dégradation

print(f"Lignes à mettre à jour : {len(update_31)}")
display(update_31.head(3))

In [ ]:
# Exécution de la mise à jour 3.1
success_31 = updater.update_database(
    update_df=update_31,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep='last',
    use_transaction=True,
)
print(f"Mise à jour 3.1 réussie : {success_31}")

# Vérification : métadonnées et tables de dimension inchangées
df_meta_post = conn.execute("SELECT name, is_categorical FROM metadata").fetchdf()
all_tables_post = [row[0] for row in conn.execute("SHOW TABLES").fetchall()]
dim_tables_post = sorted([t for t in all_tables_post if t.startswith('dim_')])
print(f"Tables de dimension après mise à jour 3.1 : {dim_tables_post}")

### 3.2. Ajout de nouvelles observations <a id="section_3_2"></a>

Insertion de 20 nouvelles lignes correspondant à une nouvelle fenêtre temporelle (2025).
Ces lignes n'existent pas dans la base : l'upsert les insère sans modifier les lignes existantes.

In [ ]:
# Génération de 20 nouvelles lignes avec des dates en 2025
np.random.seed(10)
new_start_date = datetime(2025, 1, 1)

new_rows = []
for i in range(20):
    new_rows.append({
        'indicator':     np.random.choice(indicators),
        'country':       np.random.choice(countries),
        'kind':          np.random.choice(kinds),
        'model':         np.random.choice(models),
        'training':      np.random.choice(trainings),
        'horizon':       np.random.choice(horizons),
        'date':          new_start_date + timedelta(days=i),
        'value':         np.random.uniform(10, 100),
        'lower_bound':   np.random.uniform(5,  50),
        'upper_bound':   np.random.uniform(50, 150),
        'quality_score': np.random.uniform(0.5, 1.0),
    })

df_new = pd.DataFrame(new_rows)
df_new['date'] = pd.to_datetime(df_new['date'])
df_new = df_new.drop_duplicates(subset=PK_COLUMNS, keep='first').reset_index(drop=True)

n_before = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
print(f"Lignes dans fact_table avant insertion : {n_before}")
print(f"Nouvelles observations à insérer       : {len(df_new)}")
display(df_new.head(3))

In [ ]:
# Exécution de la mise à jour 3.2 — insertion des nouvelles lignes
success_32 = updater.update_database(
    update_df=df_new,
    check_duplicates_db=True,
    check_duplicates_update=True,
    keep='last',
    use_transaction=True,
)
n_after = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
print(f"Mise à jour 3.2 réussie : {success_32}")
print(f"Lignes dans fact_table après insertion : {n_after}  (+{n_after - n_before})") 

## 4. Suppression d'observations <a id="section_4"></a>

`DatabaseDeleter.delete_rows()` accepte une liste de filtres de la forme `(colonne, opérateur, valeur)`.  
Le paramètre `perform_cleanup=True` déclenche le nettoyage des orphelins dans les tables de dimension  
et la réévaluation du statut catégoriel après la suppression.

In [ ]:
# Initialisation du composant de suppression
# La connexion est partagée avec l'updater — les opérations portent sur le même état
deleter = DatabaseDeleter(
    connection=conn,
    categorical_threshold=CATEGORICAL_THRESHOLD,
)

### 4.1. Suppression simple sans impact dimensionnel <a id="section_4_1"></a>

Suppression des observations de type `forecast` avec `horizon = 14`.  
Les tables de dimension ne sont pas affectées : `kind` et `horizon` ont d'autres modalités dans la base après la suppression.

In [ ]:
# Récupération des identifiants numériques pour les filtres catégoriels
# fact_table stocke des IDs → les filtres doivent utiliser ces IDs, pas les labels
forecast_id = conn.execute(
    "SELECT value FROM dim_kind WHERE label = 'forecast'"
).fetchone()[0]

horizon_14_id = 14

print(f"ID de 'forecast' dans dim_kind    : {forecast_id}")
print(f"ID de '14' dans dim_horizon       : {horizon_14_id}")

n_before_41 = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
n_target_41 = conn.execute(
    f"SELECT COUNT(*) FROM fact_table WHERE kind = {forecast_id} AND horizon = {horizon_14_id}"
).fetchone()[0]
print(f"Lignes cibles (forecast + horizon 14) : {n_target_41}")

In [ ]:
# Exécution de la suppression 4.1
filters_41 = [
    ('kind',    '=', forecast_id),
    ('horizon', '=', horizon_14_id),
]
deleted_41 = deleter.delete_rows(
    filters=filters_41,
    use_transaction=True,
    perform_cleanup=True,
)
n_after_41 = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
print(f"Lignes supprimées : {deleted_41}")
print(f"Lignes restantes  : {n_after_41}  (avant : {n_before_41})")

# Vérification : 'forecast' et le label 14 sont encore présents dans leurs tables de dimension
print("\nContenu de dim_kind (inchangée) :")
display(conn.execute("SELECT * FROM dim_kind ORDER BY value").fetchdf())

### 4.2. Suppression avec nettoyage dimensionnel <a id="section_4_2"></a>

Suppression de **toutes** les lignes correspondant au pays `Italy`.  
Après la suppression, `Italy` n'est plus référencé dans `fact_table` : `perform_cleanup=True`  
provoque la suppression de cette modalité orpheline dans `dim_country`.

In [ ]:
# Récupération de l'ID de Italy dans la table de dimension country
italy_id = conn.execute(
    "SELECT value FROM dim_country WHERE label = 'Italy'"
).fetchone()[0]
print(f"ID de 'Italy' dans dim_country : {italy_id}")

print("\nContenu de dim_country avant suppression :")
display(conn.execute("SELECT * FROM dim_country ORDER BY value").fetchdf())

n_italy = conn.execute(
    f"SELECT COUNT(*) FROM fact_table WHERE country = {italy_id}"
).fetchone()[0]
print(f"Lignes Italian à supprimer : {n_italy}")

In [ ]:
# Exécution de la suppression 4.2
filters_42 = [('country', '=', italy_id)]
deleted_42 = deleter.delete_rows(
    filters=filters_42,
    use_transaction=True,
    perform_cleanup=True,
)
print(f"Lignes supprimées : {deleted_42}")

# Vérification du nettoyage : Italy doit avoir disparu de dim_country
print("\nContenu de dim_country après suppression (Italy absent) :")
display(conn.execute("SELECT * FROM dim_country ORDER BY value").fetchdf())

## 5. Audit de la base de données <a id="section_5"></a>

`DatabaseAuditor` valide la cohérence du schéma, l'intégrité référentielle entre `fact_table`  
et les tables de dimension, ainsi que les problèmes de performance potentiels.  
Trois niveaux de validation sont disponibles : `BASIC`, `STANDARD`, `COMPREHENSIVE`.

In [ ]:
# Initialisation de l'auditeur avec la connexion partagée
auditor = DatabaseAuditor(
    connection=conn,
    categorical_threshold=CATEGORICAL_THRESHOLD,
)

### 5.1. Audit basique <a id="section_5_1"></a>

Le niveau `BASIC` vérifie uniquement l'existence des tables obligatoires (`fact_table`, `metadata`)  
et la cohérence minimale du schéma. Rapide, adapté aux contrôles de santé en production.

In [ ]:
# Exécution de l'audit basique
report_basic = auditor.validate_database(ValidationLevel.BASIC)

print(f"Niveau d'audit       : {report_basic.validation_level.value}")
print(f"Problèmes détectés   : {len(report_basic.issues)}")
print(f"Problèmes critiques  : {report_basic.get_critical_issues_count()}")
print(f"Résumé               : {report_basic.validation_summary}")

### 5.2. Audit standard <a id="section_5_2"></a>

Le niveau `STANDARD` ajoute la vérification de l'intégrité référentielle (pas de clé étrangère orpheline)  
et la cohérence des métadonnées (types, statut catégoriel).

In [ ]:
# Exécution de l'audit standard
report_standard = auditor.validate_database(ValidationLevel.STANDARD)

print(f"Niveau d'audit       : {report_standard.validation_level.value}")
print(f"Problèmes détectés   : {len(report_standard.issues)}")
print(f"Recommandations      : {report_standard.recommendations}")

# Détail des problèmes détectés
if report_standard.issues:
    print("\nDétail des problèmes :")
    for issue in report_standard.issues:
        print(f"  [{issue.severity.value.upper()}] {issue.table_name} — {issue.description}")
else:
    print("Aucun problème détecté à ce niveau.")

### 5.3. Audit complet <a id="section_5_3"></a>

Le niveau `COMPREHENSIVE` inclut en plus la vérification des performances (indexes, fragmentation)  
et un contrôle approfondi des données (valeurs nulles sur les clés primaires, cohérence des types).

In [ ]:
# Exécution de l'audit complet
report_comprehensive = auditor.validate_database(ValidationLevel.COMPREHENSIVE)

print(f"Niveau d'audit       : {report_comprehensive.validation_level.value}")
print(f"Tables validées      : {report_comprehensive.tables_validated}")
print(f"Résumé               : {report_comprehensive.validation_summary}")

# Regroupement des problèmes par sévérité pour faciliter la lecture
for severity_label in ['critical', 'high', 'medium', 'low']:
    count = report_comprehensive.validation_summary.get(f'{severity_label}_issues', 0)
    if count > 0:
        print(f"\nProblèmes [{severity_label.upper()}] ({count}) :")
        from dt_ducklake_manager.maintenance.auditor import IssueSeverity
        sev = IssueSeverity(severity_label)
        for issue in report_comprehensive.get_issues_by_severity(sev):
            print(f"  - {issue.table_name} : {issue.description}")

if not report_comprehensive.issues:
    print("Base de données saine — aucun problème détecté.")

## 6. Maintenance physique (DuckLake) <a id="section_6"></a>

Chaque INSERT / UPDATE / DELETE dans DuckLake produit un petit fichier Parquet ou un fichier de  
tombstones. `DuckLakeMaintenance` permet de consolider ces fichiers pour maintenir des performances  
de lecture optimales.

In [ ]:
# Initialisation du composant de maintenance avec la connexion partagée
maint = DuckLakeMaintenance(connection=conn)

### 6.1. Opérations individuelles <a id="section_6_1"></a>

Les quatre opérations DuckLake peuvent être exécutées séparément selon le besoin :
- `merge_files` : fusion des petits fichiers Parquet adjacents
- `rewrite_data_files` : suppression physique des tombstones (lignes effacées)
- `expire_snapshots` : expiration des anciens snapshots selon un seuil en jours
- `cleanup_files` : suppression des fichiers Parquet orphelins (non référencés par un snapshot actif)

In [ ]:
# Fusion des petits fichiers Parquet pour réduire le nombre de handles d'I/O
maint.merge_files(schema='main', table='fact_table')
print("merge_files terminé.")

In [ ]:
# Réécriture des fichiers pour supprimer physiquement les tombstones de suppression
maint.rewrite_data_files(schema='main', table='fact_table')
print("rewrite_data_files terminé.")

In [ ]:
# Expiration des snapshots vieux de plus de 0 jours (conserve uniquement le snapshot courant)
# En production, on utilise typiquement older_than_days=30 pour conserver un mois d'historique
maint.expire_snapshots(schema='main', older_than_days=0)
print("expire_snapshots terminé (older_than_days=0 pour illustration).")

In [ ]:
# Suppression des fichiers Parquet orphelins après expiration des snapshots
maint.cleanup_files(schema='main')
print("cleanup_files terminé.")

### 6.2. Maintenance complète <a id="section_6_2"></a>

`full_maintenance()` exécute les quatre opérations dans l'ordre recommandé.  
C'est la méthode à appeler en production après un lot de mises à jour intensif.

In [ ]:
# Simulation de quelques opérations supplémentaires pour accumuler des fichiers delta
np.random.seed(99)
extra_rows = df_new.head(5).copy()
extra_rows['value'] = extra_rows['value'] + 5
updater.update_database(extra_rows, check_duplicates_db=False, check_duplicates_update=False)
print("Mise à jour intermédiaire effectuée — fichiers delta générés.")

# Exécution de la maintenance complète avec conservation de 30 jours d'historique
maint.full_maintenance(schema='main', table='fact_table', older_than_days=30)
print("Maintenance complète terminée.")

## 7. Voyage dans le temps (time-travel) <a id="section_7"></a>

DuckLake conserve un historique complet des snapshots. Le time-travel s'effectue via la  
clause SQL `AT` sur la **connexion existante**, car DuckLake verrouille le fichier catalogue  
au niveau processus et interdit deux connexions simultanées sur le même fichier.  
`DuckLakeConnector.at_clause()` génère la clause `AT (VERSION => n)` ou `AT (TIMESTAMP => ...)`  
à insérer directement dans les requêtes `FROM`.

> **Note** : les snapshots expirés en Section 6.1 ne sont plus accessibles par time-travel.  
> Dans ce notebook, le snapshot de référence `snapshot_before_update` a été capturé **avant**  
> l'étape d'expiration ; il n'est donc disponible que si `older_than_days=0` n'a pas encore  
> purgé l'historique complet. L'exemple ci-dessous reste illustratif.

### 7.1. Consultation d'un snapshot par version <a id="section_7_1"></a>

In [ ]:
# Liste des snapshots disponibles dans le catalogue
df_snapshots = conn.execute(
    "SELECT snapshot_id, snapshot_time FROM ducklake_snapshots('db') ORDER BY snapshot_id"
).fetchdf()
print(f"Snapshots disponibles : {len(df_snapshots)}")
display(df_snapshots)

In [ ]:
# Snapshot courant avant la mise à jour — pour le voyage dans le temps en Section 7
snapshot_before_update = conn.execute(
    "SELECT snapshot_id FROM ducklake_snapshots('db') ORDER BY snapshot_id DESC LIMIT 1"
).fetchone()[0]
print(f"Snapshot de référence avant la mise à jour : {snapshot_before_update}")

In [ ]:
# Time-travel par numéro de version via la clause AT sur la connexion courante
# DuckLake interdit d'attacher le même fichier catalogue deux fois dans le même processus :
# at_clause() génère la clause SQL AT (VERSION => n) à utiliser directement dans FROM.
if snapshot_before_update in df_snapshots['snapshot_id'].values:
    connector_past = DuckLakeConnector(
        CATALOG_PATH,
        DATA_PATH,
        snapshot_version=snapshot_before_update,
    )
    at = connector_past.at_clause()

    n_past    = conn.execute(f"SELECT COUNT(*) FROM fact_table {at}").fetchone()[0]
    n_current = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]

    print(f"Lignes dans le snapshot {snapshot_before_update} (avant mises à jour) : {n_past}")
    print(f"Lignes dans la version courante                                        : {n_current}")
    print(f"Différence                                                             : {n_current - n_past}")
else:
    print(
        f"Le snapshot {snapshot_before_update} a été expiré à l'étape 6.1."
        "\nPour préserver l'historique, utiliser older_than_days > 0 dans expire_snapshots()."
    )

### 7.2. Consultation d'un snapshot par horodatage <a id="section_7_2"></a>

`snapshot_time` accepte une chaîne ISO-8601 et ouvre le snapshot le plus récent  
antérieur ou égal à l'horodatage fourni. Utile pour auditer l'état de la base  
à une date précise sans avoir à connaître le numéro de version.

In [ ]:
# Time-travel par horodatage via la clause AT sur la connexion courante
# On utilise l'horodatage du snapshot de référence (snapshot_before_update) plutôt que
# le tout premier snapshot (id=0), qui précède la création de fact_table.
if not df_snapshots.empty:
    ref_row = df_snapshots[df_snapshots['snapshot_id'] == snapshot_before_update]
    if ref_row.empty:
        print(
            f"Le snapshot {snapshot_before_update} a été expiré à l'étape 6.1."
            "Pour préserver l'historique, utiliser older_than_days > 0 dans expire_snapshots()."
        )
    else:
        snapshot_time_str = str(ref_row.iloc[0]['snapshot_time'])
        print(f"Horodatage du snapshot de référence ({snapshot_before_update}) : {snapshot_time_str}")

        connector_time = DuckLakeConnector(
            CATALOG_PATH,
            DATA_PATH,
            snapshot_time=snapshot_time_str,
        )
        at = connector_time.at_clause()

        try:
            n_time = conn.execute(f"SELECT COUNT(*) FROM main.fact_table {at}").fetchone()[0]
            print(f"Lignes dans le snapshot à {snapshot_time_str} : {n_time}")
        except Exception as e:
            print(f"Time-travel par horodatage indisponible : {e}")
else:
    print("Aucun snapshot disponible dans le catalogue.")

In [ ]:
# Fermeture propre de la connexion principale
conn.close()
print("Connexion principale fermée.")